In [2]:
!pip install -q langchain-community langchain-text-splitters pypdf

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

files = {
    "podcast": Path("data/2022_016_100_886527.pdf"),
    "eu_ai_act": Path("data/eli_reg_2024_1689_oj_EN_TXT.pdf"),
}

for source, path in files.items():
    if not path.exists():
        print(f"MISSING: {path}")
        continue

    pages = PyPDFLoader(str(path)).load()
    print(f"{source}: {len(pages)} pages")
    print(pages[0].page_content[:250], "\n")

podcast: 10 pages
Trust and AI Systems, page 1  www.sei.cmu.edu/podcasts 
Trust and AI Systems  
featuring Carol Smith and Dustin Updyke  
---------------------------------------------------------------------------------------------- 
Welcome to the SEI Podcast Series 

eu_ai_act: 144 pages
REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1 



In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

all_pages = []

for source, path in files.items():
    pages = PyPDFLoader(str(path)).load()

    for doc in pages:
        page_number = doc.metadata.get("page", 0) + 1

        if source == "podcast":
            source_type = "podcast_transcript"
            section = "transcript"
        else:
            source_type = "regulation"
            section = "EU AI Act text"

        doc.metadata.update({
            "source_type": source_type,
            "section": section,
            "page_number": page_number,
        })

    all_pages.extend(pages)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(all_pages)

print("Total chunks:", len(chunks))
print("First chunk preview:", chunks[0].page_content[:300])
print("First chunk metadata:", chunks[0].metadata)

Total chunks: 803
First chunk preview: Trust and AI Systems, page 1  www.sei.cmu.edu/podcasts 
Trust and AI Systems  
featuring Carol Smith and Dustin Updyke  
---------------------------------------------------------------------------------------------- 
Welcome to the SEI Podcast Series, a production of the Carnegie Mellon University S
First chunk metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2022-08-24T15:44:54+00:00', 'author': 'hap', 'moddate': '2022-08-24T15:44:54+00:00', 'source': 'data\\2022_016_100_886527.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_type': 'podcast_transcript', 'section': 'transcript', 'page_number': 1}


In [4]:
!pip install -q langchain-openai python-dotenv

import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found in .env")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
sample_vector = embeddings.embed_query(chunks[0].page_content)

print("Embedding dimensions:", len(sample_vector))

Embedding dimensions: 1536


In [5]:
!pip install -qU langchain-pinecone

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

if not os.getenv("PINECONE_API_KEY"):
    raise ValueError("PINECONE_API_KEY was not found in .env")

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "trustworthy-ai-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

chunk_ids = [f"lab-chunk-{i}" for i in range(len(chunks))]
vector_store.add_documents(documents=chunks, ids=chunk_ids)

print(f"Added {len(chunks)} chunks to Pinecone.")

c:\Users\Roodabeh Seif\miniconda3-pbe\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Added 803 chunks to Pinecone.


In [6]:
query = "What evidence and safeguards help a user assess whether a high-risk AI system is trustworthy?"

baseline_results = vector_store.similarity_search_with_score(query, k=5)

for rank, (doc, score) in enumerate(baseline_results, start=1):
    print(f"\n--- Result {rank} | score: {score:.4f} ---")
    print("Source type:", doc.metadata.get("source_type"))
    print("Page:", doc.metadata.get("page_number"))
    print(doc.page_content[:500])


--- Result 1 | score: 0.6790 ---
Source type: regulation
Page: 18.0
high-risk, considering their potentially significant impact on democracy, the rule of law, individual freedoms as well 
as the right to an effective remedy and to a fair trial. In particular, to address the risks of potential biases, errors and 
opacity, it is appropriate to qualify as high-risk AI systems intended to be used by a judicial authority or on its behalf 
to assist judicial authorities in researching and interpreting facts and the law and in applying the law to a concrete set 
of fac

--- Result 2 | score: 0.6569 ---
Source type: regulation
Page: 60.0
(e) the computational and hardware resources needed, the expected lifetime of the high-risk AI system and any necessary 
maintenance and care measures, including their frequency, to ensure the proper functioning of that AI system, including 
as regards software updates;
(f) where relevant, a description of the mechanisms included within the high-risk AI syste

In [7]:
for source_type in ["regulation", "podcast_transcript"]:
    print(f"\n===== {source_type} only =====")

    filtered_results = vector_store.similarity_search_with_score(
        query,
        k=3,
        filter={"source_type": source_type},
    )

    for rank, (doc, score) in enumerate(filtered_results, start=1):
        print(f"\nResult {rank} | score: {score:.4f}")
        print("Page:", doc.metadata.get("page_number"))
        print(doc.page_content[:400])


===== regulation only =====

Result 1 | score: 0.6790
Page: 18.0
high-risk, considering their potentially significant impact on democracy, the rule of law, individual freedoms as well 
as the right to an effective remedy and to a fair trial. In particular, to address the risks of potential biases, errors and 
opacity, it is appropriate to qualify as high-risk AI systems intended to be used by a judicial authority or on its behalf 
to assist judicial authorities

Result 2 | score: 0.6569
Page: 60.0
(e) the computational and hardware resources needed, the expected lifetime of the high-risk AI system and any necessary 
maintenance and care measures, including their frequency, to ensure the proper functioning of that AI system, including 
as regards software updates;
(f) where relevant, a description of the mechanisms included within the high-risk AI system that allows deployers to 
properly co

Result 3 | score: 0.6442
Page: 61.0
benchmarks and measurement methodologies.
3. The levels of

In [8]:
!pip install -q sentence-transformers

from sentence_transformers import CrossEncoder

print("Loading the reranker model...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

candidate_results = vector_store.similarity_search_with_score(query, k=10)
candidate_docs = [doc for doc, _ in candidate_results]

reranker_scores = reranker.predict(
    [(query, doc.page_content) for doc in candidate_docs],
    show_progress_bar=False,
)

reranked_results = sorted(
    zip(reranker_scores, candidate_docs),
    key=lambda item: float(item[0]),
    reverse=True,
)

for rank, (score, doc) in enumerate(reranked_results[:5], start=1):
    print(f"\n--- Reranked result {rank} | score: {float(score):.4f} ---")
    print("Source:", doc.metadata.get("source_type"))
    print("Page:", doc.metadata.get("page_number"))
    print(doc.page_content[:300])

Loading the reranker model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9388.63it/s]



--- Reranked result 1 | score: 3.8567 ---
Source: regulation
Page: 17.0
to a fair trial as well as the right of defence and the presumption of innocence, could be hampered, in particular, 
where such AI systems are not sufficiently transparent, explainable and documented. It is therefore appropriate to 
classify as high-risk, insofar as their use is permitted under rele

--- Reranked result 2 | score: 3.1242 ---
Source: podcast_transcript
Page: 2.0
maintained at a level that is appropriate for individuals. Then, by giving them that information, 
they can then determine if there is risk and how much risk and what the situation is. It is never a 
situation where we are going to be or should be building systems that are one hundred percent 
trust

--- Reranked result 3 | score: 2.8831 ---
Source: regulation
Page: 19.0
involve experts and external stakeholders. In identifying the reasonably foreseeable misuse of high-risk AI systems, 
the provider should cover uses of AI systems which, wh

In [9]:
print("BASELINE — before reranking")
for rank, (doc, score) in enumerate(baseline_results[:5], start=1):
    print(
        f"{rank}. {doc.metadata.get('source_type')}, "
        f"page {doc.metadata.get('page_number')}, "
        f"Pinecone score {score:.4f}"
    )
    print(doc.page_content[:180].replace("\n", " "), "\n")

print("RERANKED — after reranking")
for rank, (score, doc) in enumerate(reranked_results[:5], start=1):
    print(
        f"{rank}. {doc.metadata.get('source_type')}, "
        f"page {doc.metadata.get('page_number')}, "
        f"reranker score {float(score):.4f}"
    )
    print(doc.page_content[:180].replace("\n", " "), "\n")

BASELINE — before reranking
1. regulation, page 18.0, Pinecone score 0.6790
high-risk, considering their potentially significant impact on democracy, the rule of law, individual freedoms as well  as the right to an effective remedy and to a fair trial. In  

2. regulation, page 60.0, Pinecone score 0.6569
(e) the computational and hardware resources needed, the expected lifetime of the high-risk AI system and any necessary  maintenance and care measures, including their frequency, t 

3. podcast_transcript, page 2.0, Pinecone score 0.6457
maintained at a level that is appropriate for individuals. Then, by giving them that information,  they can then determine if there is risk and how much risk and what the situation 

4. regulation, page 61.0, Pinecone score 0.6442
benchmarks and measurement methodologies. 3. The levels of accuracy and the relevant accuracy metrics of high-risk AI systems shall be declared in the accompanying  instructions of 

5. regulation, page 22.0, Pinecone score 

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    parts = []
    for doc in docs:
        if doc.metadata.get("source_type") == "podcast_transcript":
            source = "SEI podcast transcript"
        else:
            source = "EU AI Act"

        page = doc.metadata.get("page_number")
        parts.append(f"[{source}, p. {page}]\n{doc.page_content}")

    return "\n\n".join(parts)

baseline_context = format_docs(
    [doc for doc, score in baseline_results[:5]]
)

reranked_context = format_docs(
    [doc for score, doc in reranked_results[:5]]
)

def answer_from_context(context):
    prompt = f"""
Answer the question using only the evidence below.
Cite factual claims with the source and page shown in brackets.
Separate legal requirements from the podcast's views.
If the evidence does not support a claim, say so.

Question: {query}

Evidence:
{context}
"""
    return llm.invoke(prompt).content

baseline_answer = answer_from_context(baseline_context)
reranked_answer = answer_from_context(reranked_context)

print("ANSWER WITHOUT RERANKING:\n")
print(baseline_answer)

print("\nANSWER WITH RERANKING:\n")
print(reranked_answer)

ANSWER WITHOUT RERANKING:

To assess whether a high-risk AI system is trustworthy, the following evidence and safeguards are relevant:

**Legal Requirements:**
1. **Human Oversight**: High-risk AI systems must be designed to allow effective human oversight during their use, aiming to minimize risks to health, safety, or fundamental rights (EU AI Act, p. 60.0).
2. **Accuracy Metrics**: The levels of accuracy and relevant accuracy metrics of high-risk AI systems must be declared in the accompanying instructions for use (EU AI Act, p. 61.0).
3. **Technical Robustness**: High-risk AI systems should be resilient to errors, faults, or inconsistencies, with technical and organizational measures in place to ensure robustness (EU AI Act, p. 22.0).
4. **Maintenance and Care**: The expected lifetime of the AI system and necessary maintenance measures must be described to ensure proper functioning, including software updates (EU AI Act, p. 60.0).
5. **Bias Mitigation**: High-risk AI systems must b